<a href="https://colab.research.google.com/github/muneer-ahmad10/Chatbot/blob/main/Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install streamlit
!pip install pyngrok
!pip install faiss-cpu
!pip install transformers accelerate torch

!pip install transformers accelerate sentencepiece

In [ ]:
from pypdf import PdfReader

def extract_text(pdf_file):

    reader = PdfReader(pdf_file)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    return text

In [ ]:
def create_chunks(text, chunk_size=50):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunks.append(
            text[i:i+chunk_size]
        )

    return chunks

In [ ]:
#test
text = """Artificial Intelligence is a field of computer science.

Machine Learning is a subset of AI.

Deep Learning uses neural networks.

Transformers use self attention mechanisms.

GPT is a decoder-only transformer.

BERT is an encoder-only transformer.

RAG combines retrieval and generation.

FAISS is used for vector search.

Embeddings convert text into vectors.

Sentence Transformers generate semantic embeddings."""


chunks = create_chunks(text)

print(chunks)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = model.encode(
    chunks
)

In [ ]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]


#Creates an empty FAISS database.

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

In [ ]:
print(index.ntotal)

In [ ]:
question = "What is GPT?"
question_embedding = model.encode([question])
distances, indices = index.search(
    np.array(question_embedding).astype("float32"),
    k=3
)
for idx in indices[0]:
    print(chunks[idx])

In [ ]:
# from transformers import pipeline
# # generator = pipeline(
# #     "text-generation",
# #     model="distilgpt2"
# # )

# generator = pipeline(
#     "text-generation",
#     model="google/flan-t5-base"
# )

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
retrieved_chunks = []

for idx in indices[0]:
    if idx != -1:  # Only grab valid chunk indices
        retrieved_chunks.append(chunks[idx])

In [ ]:
context = "\n".join(retrieved_chunks)

print(context)

In [ ]:
prompt = f"""
Context:
{context}

Question:
{question}

Answer in one sentence:
"""

In [ ]:
# response = generator(
#     prompt,
#     max_new_tokens=50,
#     repetition_penalty=1.2,  # Disincentivizes repeating the same words
#     do_sample=True,          # Adds a bit of randomness so it doesn't get stuck
#     temperature=0.7
# )

In [ ]:
# print(response[0]["generated_text"])

In [ ]:
# print(indices)

In [ ]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

In [ ]:
import transformers
print(transformers.__version__)

print(indices)
print(len(chunks))

In [ ]:
print(indices)
print(len(chunks))

# Better Chunking

In [ ]:
chunks = []

sentences = text.split(".")

for sentence in sentences:
    sentence = sentence.strip()

    if sentence:
        chunks.append(sentence)

In [ ]:
print(len(chunks))

In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
chunk_embeddings = model.encode(chunks)

In [ ]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

In [ ]:
question = "What is GPT?"

question_embedding = model.encode([question])

distances, indices = index.search(
    np.array(question_embedding).astype("float32"),
    k=3
)

print(indices)